In [1]:
import Pkg; Pkg.resolve()
import Pkg; Pkg.update()

  No Changes to `/workspaces/lecture-notebooks/Lecture 14/Project.toml`
  No Changes to `/workspaces/lecture-notebooks/Lecture 14/Manifest.toml`
    Updating registry at `~/.julia/registries/General.toml`
  No Changes to `/workspaces/lecture-notebooks/Lecture 14/Project.toml`
  No Changes to `/workspaces/lecture-notebooks/Lecture 14/Manifest.toml`


In [2]:
import Pkg; Pkg.activate(@__DIR__); Pkg.instantiate()

  Activating project at `/workspaces/lecture-notebooks/Lecture 14`


In [3]:
using LinearAlgebra
using ForwardDiff

In [4]:
function hat(v)
    return [0 -v[3] v[2];
            v[3] 0 -v[1];
            -v[2] v[1] 0]
end

hat (generic function with 1 method)

In [5]:
function L(q)
    s = q[1]
    v = q[2:4]
    L = [s    -v';
         v  s*I+hat(v)]
    return L
end

L (generic function with 1 method)

In [6]:
function R(q)
    s = q[1]
    v = q[2:4]
    R = [s    -v';
         v  s*I-hat(v)]
    return R
end

R (generic function with 1 method)

In [7]:
T = Diagonal([1; -ones(3)])
H = [zeros(1,3); I];

In [8]:
function G(q)
    G = L(q)*H
end

function Q(q)
    return H'*(R(q)'*L(q))*H
end

Q (generic function with 1 method)

In [9]:
J = Diagonal([1; 2; 3])
h = 0.1

0.1

In [10]:
#initial conditions
Q0 = Array(I(3))
q0 = [1; 0; 0; 0]
ω0 = randn(3)
x0 = [vec(Q0); ω0]
x0q = [q0; ω0]

7-element Vector{Float64}:
  1.0
  0.0
  0.0
  0.0
  0.111433990601636
 -0.3678542298705924
  0.5174249514414752

In [11]:
#dynamics
function dynamics(x)
    Q = reshape(x[1:9],3,3)
    ω = x[10:12]
    
    Q̇ = Q*hat(ω)
    ω̇ = -J\(hat(ω)*J*ω)

    ẋ = [vec(Q̇); ω̇]
end

dynamics (generic function with 1 method)

In [12]:
function rkstep(x)
    f1 = dynamics(x)
    f2 = dynamics(x + 0.5*h*f1)
    f3 = dynamics(x + 0.5*h*f2)
    f4 = dynamics(x + h*f3)
    xn = x + (h/6.0)*(f1 + 2*f2 + 2*f3 + f4)
    return xn
end

rkstep (generic function with 1 method)

In [13]:
xk = x0
for k = 1:10000
    xk = rkstep(xk)
end

In [14]:
Qk = reshape(xk[1:9],3,3)

3×3 Matrix{Float64}:
 0.98333   -0.178881  -0.0323915
 0.10267    0.693515  -0.713081
 0.150022   0.697873   0.700331

In [15]:
Qk'*Qk

3×3 Matrix{Float64}:
 0.999986    5.98232e-7  1.95201e-6
 5.98232e-7  0.999987    3.974e-6
 1.95201e-6  3.974e-6    0.999998

In [16]:
#quaternion dynamics
function qdynamics(x)
    q = x[1:4]
    ω = x[5:7]
    
    q̇ = 0.5*L(q)*H*ω
    ω̇ = -J\(hat(ω)*J*ω)

    ẋ = [q̇; ω̇]
end

qdynamics (generic function with 1 method)

In [17]:
function qrkstep(x)
    f1 = qdynamics(x)
    f2 = qdynamics(x + 0.5*h*f1)
    f3 = qdynamics(x + 0.5*h*f2)
    f4 = qdynamics(x + h*f3)
    xn = x + (h/6.0)*(f1 + 2*f2 + 2*f3 + f4)
    xn[1:4] .= xn[1:4]./norm(xn[1:4])
    return xn
end

qrkstep (generic function with 1 method)

In [18]:
xkq = x0q
for k = 1:10000
    xkq = qrkstep(xkq)
end

In [19]:
qk = xkq[1:4]

4-element Vector{Float64}:
  0.9188504974725769
  0.3838936707775853
 -0.04963351344310601
  0.07665459656538731

In [20]:
norm(qk)


1.0

In [21]:
Q(qk)'*Q(qk)

3×3 Matrix{Float64}:
  1.0          1.38778e-17  -1.38778e-17
  1.38778e-17  1.0           0.0
 -1.38778e-17  0.0           1.0